# Fine-Tuning GPT-2 for IMDb Movie Review Sentiment Classification

This notebook fine-tunes **GPT-2 (124M)** on the **IMDb** movie-review dataset for a
2-class sentiment classification task (Negative / Positive), then serves the model
through a **Gradio** interface.

It follows the 14 experiment steps:

1. Verify hardware and import libraries
2. Load a dataset (IMDb)
3. Load the pretrained GPT-2 tokenizer
4. Tokenize the text
5. Create smaller training and testing datasets
6. Initialize GPT-2 for classification
7. Define the evaluation metric
8. Configure fine-tuning
9. Fine-tune GPT-2
10. Evaluate the fine-tuned model
11. Save the fine-tuned model
12. Create an inference pipeline
13. Create a Gradio interface
14. Launch the local application

## Setup: install dependencies

Run this cell once (skip if the packages are already installed).

In [ ]:
# !pip install -q torch transformers datasets evaluate numpy gradio scikit-learn

## Step 1 — Verify hardware and import libraries

Import PyTorch, Hugging Face libraries (`datasets`, `evaluate`, `transformers`), NumPy and
Gradio. Check whether a local GPU is available and display its name if CUDA is present.

In [ ]:
import numpy as np
import torch
import gradio as gr

import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

# Check whether a local GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

# Display the GPU name if CUDA is available
if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (training will be slower).")

## Step 2 — Load a dataset (IMDb)

Load the `stanfordnlp/imdb` dataset. It contains movie reviews with sentiment labels
(`0 = Negative`, `1 = Positive`).

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print("\nExample review:")
print(dataset["train"][0]["text"][:400], "...")
print("Label:", dataset["train"][0]["label"])

## Step 3 — Load the pretrained GPT-2 tokenizer

Load the GPT-2 tokenizer using `AutoTokenizer`. GPT-2 has no native padding token, so we
assign the EOS token as the padding token.

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 does not have a native padding token -> use the EOS token as the pad token
tokenizer.pad_token = tokenizer.eos_token
print("Pad token:", tokenizer.pad_token, "| Pad token id:", tokenizer.pad_token_id)

## Step 4 — Tokenize the text

Convert movie reviews into token IDs with padding, truncation and a maximum sequence
length of 512 tokens.

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print(tokenized_datasets)

## Step 5 — Create smaller training and testing datasets

Shuffle the datasets and select **3,000 reviews for training** and **1,000 reviews for
testing** to make local experimentation faster.

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(3000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

print("Training samples:", len(small_train_dataset))
print("Testing samples :", len(small_eval_dataset))

## Step 6 — Initialize GPT-2 for classification

Load the pretrained GPT-2 model and add a **2-class classification head**
(0 = Negative, 1 = Positive). Configure the model's padding token to match the tokenizer.

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

# Configure the model's padding token (GPT-2 uses EOS as pad)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)
print("Model loaded with a 2-class classification head.")

## Step 7 — Define the evaluation metric

Use **accuracy** as the evaluation metric: compare the model's predicted class with the
actual IMDb label.

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

## Step 8 — Configure fine-tuning

Use Hugging Face `TrainingArguments` with the required settings:

- Learning rate = `2e-5`
- Batch size = `4`
- Epochs = `2`
- Weight decay = `0.01`
- FP16 mixed precision (enabled when a GPU is available)
- Evaluate after each epoch
- `load_best_model_at_end=True` to keep the best checkpoint based on evaluation accuracy

In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-imdb-finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # FP16 mixed precision (GPU only)
    eval_strategy="epoch",           # evaluate after each epoch
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,     # keep the best checkpoint
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
)

## Step 9 — Fine-tune GPT-2

Create a Hugging Face `Trainer` with the model, training dataset, testing dataset,
training configuration and accuracy metric, then start the training loop.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

trainer.train()

## Step 10 — Evaluate the fine-tuned model

Evaluate the model on the IMDb test dataset. Because `load_best_model_at_end=True`, the
best checkpoint (by evaluation accuracy) is already loaded.

In [ ]:
eval_results = trainer.evaluate()
print("Final evaluation results:")
for k, v in eval_results.items():
    print(f"  {k}: {v}")

## Step 11 — Save the fine-tuned model

Save the trained GPT-2 model and its tokenizer locally.

In [ ]:
save_dir = "./gpt2-imdb-sentiment"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to: {save_dir}")

## Step 12 — Create an inference pipeline

Load the saved model using the Hugging Face `pipeline` and configure it for
**sentiment analysis** (text classification).

In [ ]:
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=save_dir,
    tokenizer=save_dir,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=512,
)

# Quick sanity check
print(sentiment_pipeline("An absolute masterpiece, I loved every minute of it!"))
print(sentiment_pipeline("A boring, predictable mess. Total waste of time."))

## Step 13 — Create a Gradio interface

Provide a text box where users can enter a movie review. The model predicts
**Positive** or **Negative** and displays the prediction with its confidence score.

In [ ]:
def predict_sentiment(review):
    if not review or not review.strip():
        return "Please enter a movie review."
    result = sentiment_pipeline(review)[0]
    label = result["label"]
    # Normalise label to a readable form
    label = "Positive" if label.upper().endswith("1") or label.upper() == "POSITIVE" else "Negative"
    score = result["score"]
    return f"Prediction: {label}  (confidence: {score:.2%})"

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(lines=5, label="Movie Review", placeholder="Type a movie review here..."),
    outputs=gr.Textbox(label="Sentiment"),
    title="GPT-2 IMDb Sentiment Classifier",
    description="Fine-tuned GPT-2 predicts whether a movie review is Positive or Negative.",
    examples=[
        ["This film blew me away — brilliant acting and a gripping story."],
        ["I fell asleep halfway through. Dull and forgettable."],
    ],
)

## Step 14 — Launch the local application

Run the Gradio application so users can enter their own movie reviews and test the
fine-tuned model.

In [ ]:
demo.launch()